### Intact animals performance curves

In [ ]:
import numpy as np
import pandas as pd
import mab_subjects

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)

trial_filter = dict(min_trials=100, clip_max=100)

perf_df = []

for i, exp in enumerate(exps):

    print(exp.sub_name)
    task = exp.b2a

    if exp.data_tag != "RNNdataset":
        task.auto_block_window_ids()
        _, expert_datetime, _, _ = task.get_expertise_day(
            by="datetime",
            day_start_hour=19,
            threshold_pct=65,
            fill_missing=True,
            n_consecutive=3,
            min_trials_per_day=500,
            baseline_frac=1,  # all data
        )
        task = task.filter_by_datetime(start=expert_datetime)

    task = exp.b2a.filter_by_trials(**trial_filter)

    # Various difficulty levels and combinations

    mab_easy = task.filter_by_deltaprob(delta_min=0.38)
    mab_hard = task.filter_by_deltaprob(delta_min=0.1, delta_max=0.35)

    perf = task.get_optimal_choice_probability()
    perf_easy = mab_easy.get_optimal_choice_probability()
    perf_hard = mab_hard.get_optimal_choice_probability()

    perf_corr = perf_uncorr = np.nan
    if exp.paradigm_tag == "9505" or exp.paradigm_tag == "8020":
        probs_all = task.probs
        prob_diff = np.diff(probs_all, axis=1).squeeze().round(2)

        df = pd.DataFrame()

        corr_mask = probs_all.sum(axis=1).round(2) == 1.0  # correlated combinations
        uncorr_mask = ~corr_mask  # uncorrelated combinations
        easy_mask = prob_diff >= 0.3  # easy combinations
        hard_mask = ~easy_mask  # hard combinations

        corr_easy_mask = corr_mask & easy_mask
        corr_hard_mask = corr_mask & hard_mask

        perf_corr = task._filtered(corr_mask).get_optimal_choice_probability()
        perf_uncorr = task._filtered(uncorr_mask).get_optimal_choice_probability()

    df = pd.DataFrame(
        dict(
            trial_id=np.arange(perf.size) + 1,
            perf=perf,
            perf_easy=perf_easy,
            perf_hard=perf_hard,
            perf_corr=perf_corr,
            perf_uncorr=perf_uncorr,
            **exp.common_kwargs,
        )
    )
    perf_df.append(df)

perf_df = pd.concat(perf_df, ignore_index=True)
mab_subjects.GroupData().save(perf_df, "poster_perf_difficulty_level")

In [ ]:
# %matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns
from neuropy import plotting
import mab_subjects
import numpy as np
from mab_colors import Palette2Arm
from statplotannot.plots import SeabornPlotter, fix_legend

df = mab_subjects.GroupData().poster_perf_difficulty_level.latest

# Filter for animal data
animal_df = df[
    (df["dataset"] != "RNNdataset")
    & (df["lesion"] == "intact")
    & (df["paradigm"] == "8020")
]

# Filter for RNN data
rnn_df = df[(df["dataset"] == "RNNdataset") & (df["paradigm"] == "8020")]

palette = Palette2Arm().as_dict()

fig = plotting.Fig(10, 7, size=(36, 48), fontsize=32, axis_lw=3, tick_size=6)
hue_order = ["unstruc", "struc"]
linestyles = ["-", "--", ":"]
titles = [
    "All permutations",
    "Easy\n(DeltaP>=40)",
    "Hard\n(DeltaP<=30)",
    "Correlated",
    "Uncorrelated",
]

for d, df in enumerate([animal_df]):
    for i, y in enumerate(["perf", "perf_easy", "perf_hard"]):

        ax = fig.subplot(fig.gs[0, i])
        plot_kw = dict(data=df, x="trial_id", y=y, hue="group", ax=ax)
        sns.lineplot(
            palette=palette,
            lw=5,
            # ls=linestyles[i],
            ls="-",
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            # style="name",
            **plot_kw,
        )
        ax.set_ylim(0.45, 1)
        ax.legend_.remove()
        # fix_legend(ax, only_labels=True, fw="bold", fs=10)
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_ylabel("P(High)")
        ax.set_xlabel("Trials")
        ax.set_xticks([1, 25, 50, 75, 100])
        ax.set_title(titles[i])

figpath = mab_subjects.FigPath.posters / "perf_intact"

fig.savefig(figpath)

### Performance matrix

In [ ]:
import numpy as np
import pandas as pd
import mab_subjects

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)

prob_perf_df = []

for i, exp in enumerate(exps):
    print(exp.sub_name)
    task = exp.b2a

    if task.datetime is not None:
        start_date = task.datetime[0]
        n_days = task.n_days
        if n_days > 60:
            task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=30))

    task = task.filter_by_trials(min_trials=100, clip_max=100)

    # print(task.probs_corr)
    # task.auto_block_window_ids()
    # mask = task.block_ids == 2
    # task = task._filtered(mask).filter_by_trials(100, 100)

    perf_mat, unique_probs = task.get_performance_prob_grid(n_last_trials=90)

    df = pd.DataFrame(
        {
            "probs": [unique_probs],
            "perf_mat": [perf_mat],
            **exp.common_kwargs,
        }
    )
    prob_perf_df.append(df)

prob_perf_df = pd.concat(prob_perf_df, ignore_index=True)
mab_subjects.GroupData().save(prob_perf_df, "poster_perf_probability_matrix")

In [ ]:
import matplotlib.pyplot as plt
from neuropy.plotting import Fig
import seaborn as sns
from scipy.stats import binned_statistic_2d
from mab_colors import Palette2Arm
from statplotannot.plots import fix_legend
import mab_subjects
import numpy as np
from scipy.ndimage import gaussian_filter
from palettable.scientific.sequential import GrayC_7
from matplotlib.colors import BoundaryNorm, CenteredNorm
import matplotlib.ticker as ticker

bounds = np.linspace(0.4, 1, 15)
norm = BoundaryNorm(bounds, ncolors=256)

fig = Fig(11, 7, size=(36, 48), fontsize=32, axis_lw=3, tick_size=6)

df = mab_subjects.GroupData().poster_perf_probability_matrix.latest
df = df[df["dataset"] != "RNNdataset"]
# df = df[df["lesion"] == "pre_lesion"]
matrix = []
for g, grp in enumerate(["struc", "unstruc"]):
    df_grp = df[df["group"] == grp]
    perf_mean = df_grp["perf_mat"].mean()
    # perf_mean = gaussian_filter(perf_mean, sigma=0.1)
    # perf_mean = np.tril(perf_mean, k=0)
    mask = np.triu(np.ones_like(perf_mean, dtype=bool), k=0)

    perf_mean[mask] = np.nan
    matrix.append(perf_mean)

    ax = fig.subplot(fig.gs[g])

    im = ax.pcolormesh(
        perf_mean.T,
        # cmap=GrayC_7.mpl_colormap,
        cmap="turbo",
        shading="auto",
        # vmin=0.6,
        # vmax=1,
        norm=norm,
    )
    ticks = np.arange(0, 6) + 0.5
    ax.set_xticks(ticks, [20, 30, 40, 60, 70, 80])
    ax.set_yticks(ticks, [20, 30, 40, 60, 70, 80])
    ax.set_xlim(1, 6)
    ax.set_ylim(0, 5)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)
    ax.set_xlabel("Higher probability")
    ax.set_ylabel("Lower probability")

    cb = plt.colorbar(im, ax=ax, shrink=0.7, location="bottom")
    cb.set_label("P(High)")
    cb.set_ticks([0.5, 0.75, 1.0])
    cb.formatter = ticker.FormatStrFormatter("%.2f")
    cb.update_ticks()
    cb.ax.tick_params(labelsize=12)
    cb.outline.set_linewidth(2)

    ax.set_title(f"{grp} sessions")

matrix_diff = matrix[0] - matrix[1]

ax = fig.subplot(fig.gs[3])
im = ax.pcolormesh(
    matrix_diff.T,
    # cmap=GrayC_7.mpl_colormap,
    cmap="bwr",
    shading="auto",
    norm=CenteredNorm(vcenter=0, halfrange=0.1),
)
ticks = np.arange(0, 6) + 0.5
ax.set_xticks(ticks, [20, 30, 40, 60, 70, 80])
ax.set_yticks(ticks, [20, 30, 40, 60, 70, 80])
ax.set_xlim(1, 6)
ax.set_ylim(0, 5)
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(True)
ax.set_xlabel("Higher probability")
ax.set_ylabel("Lower probability")

cb = plt.colorbar(im, ax=ax, shrink=0.7, location="bottom")
cb.set_label("Diff.")
cb.formatter = ticker.FormatStrFormatter("%.2f")
cb.update_ticks()
cb.outline.set_linewidth(2)
# ax.set_title(f"{grp} sessions\n(mean of last 99 trials)")

figpath = mab_subjects.FigPath.posters / "perf_probability_matrix"
fig.savefig(figpath)

### Lesion performance

In [ ]:
from scipy import stats
import mab_subjects
import numpy as np
import pandas as pd

# exps = (
#     mab_subjects.unstruc.p8020_lesion_mPFC_pre_post_sess
#     + mab_subjects.struc.p8020_lesion_mPFC_pre_post_sess
# )
exps = (
    mab_subjects.unstruc.p8020_lesion_OFC_pre_post_sess
    + mab_subjects.struc.p8020_lesion_OFC_pre_post_sess
)


perf_df = []

for i, exp in enumerate(exps):
    print(exp.sub_name)

    task = exp.b2a
    start_date = task.datetime[0]
    stop_date = task.datetime[-1]
    n_days = pd.Timedelta(stop_date - start_date).days

    if n_days > 60:
        task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=40))

    # if i==1:
    # task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=3))

    probs_corr = task.probs_corr
    print(probs_corr)

    task_filt = task.filter_by_trials(min_trials=100, clip_max=100)
    perf_overall = task_filt.get_optimal_choice_probability()

    df = pd.DataFrame()
    df["trial_id"] = np.arange(perf_overall.shape[0]) + 1
    df["All"] = perf_overall
    df["name"] = exp.sub_name
    df["grp"] = exp.group_tag
    df["lesion"] = exp.lesion_tag
    perf_df.append(df)

perf_df = pd.concat(perf_df, ignore_index=True)
mab_subjects.GroupData().save(perf_df, "poster_perf_vs_ofc_lesion")

In [ ]:
from statplotannot.plots import SeabornPlotter, fix_legend
from mab_colors import Palette2Arm
import seaborn as sns
from statplotannot.plots import Fig
import mab_subjects
from statplotannot.plots import Fig

perf_df = mab_subjects.GroupData().poster_perf_vs_mpfc_lesion.latest


fig = Fig(7, 5, size=(36, 48), fontsize=40, axis_lw=5, tick_size=7)

for g, grp in enumerate(["unstruc", "struc"]):
    perf_df_grp = perf_df[perf_df["grp"] == grp]
    for i, y in enumerate(["All"]):
        ax = fig.subplot(fig.gs[g])
        sns.lineplot(
            data=perf_df_grp,
            x="trial_id",
            y=y,
            hue="lesion",
            palette=(
                Palette2Arm().unstruc_lesion_vs_intact()
                if grp == "unstruc"
                else Palette2Arm().struc_lesion_vs_intact()
            ),
            errorbar="se",
            lw=9,
            err_kws=dict(edgecolor="none", alpha=0.3),
        )
        fix_legend(ax)
        ax.set_ylim(0.4, 0.9)
        ax.set_ylabel("P(High)")
        ax.set_title(f"{grp}")
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        fix_legend(ax, only_labels=True)

figpath = mab_subjects.FigPath.posters / "perf_vs_ofc_lesion"
fig.savefig(figpath)

### Qlearn easy hard

In [ ]:
import numpy as np
import pandas as pd
import mab_subjects
from statplotannot.plots import SeabornPlotter, Fig, xtick_format
from mab_colors import Palette2Arm

data_df = mab_subjects.GroupData().fit_qlearn_easy_hard.latest

# --- policy filter ---------
policies = data_df["policy"].unique()
fit_scopes = data_df["fit_scope"].unique()
policy_select = "Qlearn2Arm"
data_df = data_df[data_df["policy"].isin([policy_select])]
policy_params = data_df["param_names"].unique()

match policy_select:
    case "Qlearn2Arm":
        policy_params_select = ["alpha_c", "alpha_u"]
    case "QlearnBias2Arm":
        policy_params_select = ["alpha_c", "alpha_u", "b0"]
    case "QlearnHierarchical2Arm":
        policy_params_select = ["alpha_c", "alpha_u", "b0", "beta"]
    case "QlearnAdaptiveLR":
        policy_params_select = ["alpha_c0", "alpha_u0", "kappa_c", "kappa_u", "beta"]
    case "QlearnAdaptiveLR":
        policy_params_select = ["alpha_c0", "alpha_u0", "kappa_c", "kappa_u", "beta"]
    case "QlearnDiff":
        policy_params_select = ["alpha", "bias", "beta"]


n_params = len(policy_params_select)

rnn_dataset = data_df[data_df["dataset"] == "RNNdataset"]
animal_dataset = data_df[data_df["dataset"] != "RNNdataset"]

palette = Palette2Arm().as_dict()
fig = Fig(1, 1, size=(36, 48), axis_lw=5, tick_size=7, fontsize=40)

for d, dataset in enumerate([animal_dataset]):

    df = dataset.copy()
    dataset_name = "RNN" if d == 0 else "Animal"
    df = df[df["param_names"].isin(policy_params_select)]
    fit_scope = df["fit_scope"].unique()

    subfig = fig.add_subfigure(fig.gs[d])
    subfig.suptitle(f"{dataset_name} Dataset", fontsize=40)
    axs = subfig.subplots(len(fit_scope) + 6, n_params + 6)

    # for s, scope in enumerate(fit_scope):
    # df_scope = df[df["fit_scope"] == scope]
    # df_scope["param_values"] = df_scope["param_values"].astype(float)
    df["param_values"] = df["param_values"].astype(float)

    for i, param in enumerate(policy_params_select):
        df_param = df[df["param_names"] == param]
        ax = axs[0, i]

        SeabornPlotter(
            data=df_param,
            x="fit_scope",
            y="param_values",
            hue="group",
            hue_order=["unstruc", "struc"],
            ax=ax,
        ).boxplot_filled(palette=palette).stat_test(test_name="Mann-Whitney-gt")

        ax.set_xlabel("")

        if "stay" in param:
            ax.set_ylim(0, 1)

        if i == 0:
            ax.set_title(f"{dataset_name} ({scope})")

            # ax.set_ylim(-0.1, 0.4)
            # ax.set_yticks([-0.1, 0, 0.1, 0.2, 0.3, 0.4])
            pass
            # if i != 0:
            # ax.set_ylim(-0.2, 0.05)
            # ax.set_ylabel("")

figpath = mab_subjects.FigPath.posters / "qlearn_easy_hard"
fig.savefig(figpath)

### Switch probability trial wise

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from neuropy import plotting
import mab_subjects
import numpy as np
from statannotations.Annotator import Annotator
from statplot_utils import stat_kw
from mab_colors import Palette2Arm

fig = plotting.Fig(9, 5, size=(36, 48), fontsize=40, axis_lw=5, tick_size=7)

grpdata = mab_subjects.GroupData()
df = grpdata.switch_prob_by_trial_100trials.latest
# df = df[df["first_experience"] == True]

for b, block in enumerate(["all", "block1", "block2plus"]):
    df_block = df[df["block_type"] == block]
    ax = fig.subplot(fig.gs[b])

    plot_kw = dict(
        data=df_block,
        x="trial_id",
        y="switch_prob",
        hue="group",
        hue_order=["unstruc", "struc"],
        ax=ax,
    )
    sns.lineplot(
        palette=Palette2Arm().as_dict(),
        lw=7,
        errorbar="se",
        err_kws={"edgecolor": None, "alpha": 0.3},
        **plot_kw,
    )

    ax.set_title("Switch probability")
    ax.get_legend().remove()
    # ax.set_ylim(0, 0.25)
    ax.set_xlim(1, 100)
    ax.set_xticks([1, 25, 50, 75, 100])

figpath = mab_subjects.FigPath.posters / "switch_prob_by_trial"
fig.savefig(figpath)